# Kimi K3 via the direct Moonshot API

Runs annotation through Moonshot's first-party endpoint instead of
OpenRouter. Why: first-party serving has no 64K completion cap
(`max_completion_tokens` defaults to 131,072, settable to 1M), full
precision, and automatic prefix caching that discounts the repeated P1
system prompt. Logic lives in
`extension/scripts/model_specific/moonshot_kimi.py`; cache, validation,
and scoring are shared. Records cache under
`moonshot-direct/kimi-k3-{effort}`, separate from OpenRouter K3 records.

Sampling is fixed server-side (temperature 1.0, top_p 0.95): K3 remains
stochastic across runs on this backend, per Moonshot's documentation.
Effort levels: low, high, max (max is the platform default and the
configuration the OpenRouter sweeps effectively ran).

**Before running:** put `MOONSHOT_API_KEY=...` in the environment or in
`.env` at the repo root. K3 unlocks after a minimum $1 top-up; rate
limits scale with the account tier, so keep workers modest.


In [1]:
import os, sys
from pathlib import Path
_here = Path.cwd()
for _c in [_here, *_here.parents]:
    if (_c / "extension" / "artifacts").exists():
        os.chdir(_c); break
sys.path.insert(0, str(Path.cwd()))
from extension.scripts.model_specific import moonshot_kimi
try:
    moonshot_kimi._api_key(); _key = True
except RuntimeError:
    _key = False
print("cwd:", os.getcwd(), "| MOONSHOT key found:", _key)


cwd: /Users/tandon.utsav2/Desktop/Experiment_1 | MOONSHOT key found: True


In [2]:
from extension.scripts.load_annotation_data import load_dataset
from extension.scripts import prompt_loader, extraction, scoring

gold = load_dataset("extension/artifacts/annotation_dev_and_val_sets/validation_set.csv")
DIALOGUES = extraction.dialogues_from(gold, split="train")
print(f"{len(DIALOGUES)} dialogues, {len(gold)} units")


78 dialogues, 544 units


In [3]:
TEST_PROMPTS = ['P1_full_codebook']
N_TEST_DIALOGUES = 78
MAX_WORKERS = 10
EFFORT = 'max'        # K3 scale: 'low', 'high', 'max' (platform default)


In [4]:
TEST_DIALOGUES = DIALOGUES[:N_TEST_DIALOGUES]
for _p in TEST_PROMPTS:
    assert _p in prompt_loader.list_prompts(), f"unknown prompt {_p!r}"
SLUG = moonshot_kimi.cache_slug(EFFORT)
print(f"moonshot-direct kimi-k3 effort={EFFORT} on "
      f"{[d['dialogue_id'] for d in TEST_DIALOGUES]} x {TEST_PROMPTS}\n")

import json as _json
import pandas as pd
family_f1_cols = [f'f1_{family}' for family in scoring.FAMILIES]
test_rows = []
for prompt in TEST_PROMPTS:
    print(f"  {prompt} ({len(TEST_DIALOGUES)} dialogues, {MAX_WORKERS} workers):")
    moonshot_kimi.generate_annotations(prompt, TEST_DIALOGUES,
                                       reasoning_effort=EFFORT, max_workers=MAX_WORKERS)
    for dlg in TEST_DIALOGUES:
        rec = _json.load(open(extraction.cache_path(SLUG, prompt,
                                                    dlg['dialogue_id'], dlg['split'])))
        u = (rec['attempts'][0].get('meta') or {}).get('usage', {}) if rec['attempts'] else {}
        print(f"  {prompt:22s} {dlg['dialogue_id']}: {'ok' if rec['valid'] else 'invalid':8s} "
              f"completion {u.get('completion_tokens','?')} tok  latency {rec['latency_s']:.1f}s")
    s = scoring.score_config(gold, SLUG, prompt,
                             [d['dialogue_id'] for d in TEST_DIALOGUES], n_boot=0,
                             split='train')
    test_rows.append(s)
    print(f"  -> validity {s['valid_rate']:.0%} | macro-F1(P) {s['macro_f1_P']:.3f} "
          f"| micro-F1(P) {s['micro_f1_P']:.3f} "
          f"| weighted-F1(P) {s['weighted_f1_P']:.3f} "
          f"| alpha {s['alpha']:.3f}")
    print("     family F1(P): " + " | ".join(
        f"{family}={s[f'f1_{family}']:.3f}" for family in scoring.FAMILIES
    ) + "\n")

summary_cols = ['prompt', 'valid_rate', 'macro_f1_P', 'micro_f1_P',
                'weighted_f1_P', 'alpha',
                *family_f1_cols, 'latency_s']
summary = pd.DataFrame(test_rows)[summary_cols].round(3)
print(summary.to_string(index=False))


moonshot-direct kimi-k3 effort=max on [1, 21, 35, 79, 143, 178, 255, 270, 275, 289, 300, 306, 323, 344, 351, 356, 380, 434, 448, 494, 532, 554, 589, 617, 635, 656, 695, 736, 758, 779, 818, 822, 842, 862, 947, 958, 966, 980, 992, 1026, 1051, 1063, 1071, 1084, 1089, 1107, 1119, 1217, 1221, 1300, 1349, 1421, 1452, 1490, 1519, 1540, 1553, 1555, 1557, 1571, 1615, 1658, 1668, 1717, 1719, 1776, 1780, 1870, 1890, 1916, 1941, 2018, 2164, 2192, 2196, 2202, 2206, 2222] x ['P1_full_codebook']

  P1_full_codebook (78 dialogues, 10 workers):
  79: cached
  178: cached
  35: cached
  143: cached
  1: cached
  270: cached
  21: cached
  255: cached
  289: cached
  300: cached
  356: cached
  532: cached
  380: cached
  494: cached
  554: cached
  589: cached
  617: cached
  656: cached
  635: cached
  695: cached
  758: cached
  779: cached
  736: cached
  818: cached
  822: cached
  862: cached
  842: cached
  947: cached
  966: cached
  958: cached
  980: cached
  992: cached
  1026: cached
  1051: 

### Notes

Cost prints are omitted: the direct API reports tokens (with cache-hit
accounting on billing), not dollars. Prefix caching is automatic, so the
40K-token P1 system prompt should show as cache hits from the second
dialogue onward. Invalid records re-fire on the next execution; the purge
script covers this cache tree too. For a deadlock cell, raising
`moonshot_kimi.MAX_COMPLETION_TOKENS` (up to 1,048,576) buys headroom.
